In [34]:
import os
import pandas as pd
from khiops import core as kh
from khiops.sklearn import KhiopsClassifier, train_test_split_dataset
from sklearn import metrics
import numpy as np
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

# We here use the khiops library: www.khiops.org <br>
> Two data files are required: <br>
    > <ul> S_PatientsWithLabels.csv and S_PatientWithFeatures.csv <br></ul>
> and one dictionary file: <br>
> <ul>Sepsis.kdic <br></ul>
> all the 3 files are available is this Github <br><br>
> Note : the two data files must be sorted (alphanumeric sort) according to the column 'Id' wihch is here the case <br>
> this sort may be realised using the khiops library (see the doc on www.khiops.org) <br>


In [35]:
# Load the root data table and the secondary data table

In [37]:
S_PatientsWithLabels = pd.read_csv("S_PatientsWithLabels-sep-tab.txt", sep="\t")
S_PatientsWithFeatures_4logs = pd.read_csv("S_PatientsWithFeatures_4logs_sep-tab.txt", sep="\t")

In [38]:
# Before training the classifier, we split the the root data into a feature matrix (only the Id column)
# and a target vector containing the labels (the Target column).

In [39]:
Id_patients = S_PatientsWithLabels.drop("Target", axis=1)
y_train = S_PatientsWithLabels["Target"]

In [40]:
# nb examples in the root data table
N = len(Id_patients)

In [41]:
# define the relational schema for the khiops classifier

In [42]:
X = {
    "main_table": (Id_patients, ["Patient_ID"]),
    "additional_data_tables": {
        "S_PatientsWithFeatures_4logs": (S_PatientsWithFeatures_4logs, ["Patient_ID"]),
    },
}

In [43]:
# Split the dataset into train and test
X_train, X_test, y_train, y_test = train_test_split_dataset(
    X, y_train, test_size=0.1, random_state=1
)


In [44]:
# Call to the KhiopsClassifier fit method

In [45]:
khc_sepsis = KhiopsClassifier(n_features=1000, n_trees=0)
khc_sepsis.fit(X_train, y_train)

KhiopsClassifier(n_features=1000, n_trees=0)

In [46]:
# performances on train

In [47]:
sepsis_train_performance = (
    khc_sepsis.model_report_.train_evaluation_report.get_snb_performance()
)
print(f"Train accuracy: {sepsis_train_performance.accuracy}")
print(f"Train auc     : {sepsis_train_performance.auc}")

Train accuracy: 0.952538
Train auc     : 0.991239


In [48]:
# deploy the model on test to have test performances

In [49]:
y_sepsis_test_predicted = khc_sepsis.predict(X_test)
probas_test = khc_sepsis.predict_proba(X_test)

In [50]:
test_accuracy = metrics.accuracy_score(y_test, y_sepsis_test_predicted)
test_auc = metrics.roc_auc_score(y_test, probas_test[:, 1])
print(f"Test accuracy: {test_accuracy}")
print(f"Test auc     : {test_auc}")

Test accuracy: 0.91324200913242
Test auc     : 0.9747482119398628


In [51]:
# number of univriate variable with level > 0
R=khc_sepsis.model_report_.preparation_report.informative_variable_number
print(f"Number of informative variables: {R}")

Number of informative variables: 865


In [52]:
# number of variable selected in the final model
S=khc_sepsis.model_report_.modeling_report.trained_predictors[0].variable_number
print(f"Number of variable selected in the final model: {S}")

Number of variable selected in the final model: 113
